# Energy Consumption Forecasting in Buildings

**Author:** Jesutomito Morakinyo  
This notebook preserves the completed coursework analysis and saved outputs.

## 1. Dataset exploration, correlation analysis and VIF

In [1]:
"""Dataset exploration, correlation analysis and VIF calculation."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.impute import KNNImputer
from statsmodels.stats.outliers_influence import variance_inflation_factor


DATA_PATH = Path("../data/processed/data_merged_data7.csv")
RESULTS_FOLDER = Path("../results")
DATE_COLUMN = "date"
TARGET_COLUMNS = ["mels_S", "lig_S", "mels_N", "hvac_N", "hvac_S"]


def main() -> None:
    merged_df = pd.read_csv(DATA_PATH)
    RESULTS_FOLDER.mkdir(exist_ok=True)

    print("Dataset shape:", merged_df.shape)
    print("\nData types:")
    print(merged_df.dtypes)
    print("\nSummary statistics:")
    print(merged_df.describe(include="all").T)
    print("\nMissing values:")
    print(merged_df.isna().sum().sort_values(ascending=False).head(30))

    if DATE_COLUMN in merged_df.columns:
        merged_df[DATE_COLUMN] = pd.to_datetime(
            merged_df[DATE_COLUMN], errors="coerce"
        )
        merged_df = merged_df.sort_values(DATE_COLUMN)

        plt.figure(figsize=(14, 7))
        for target in TARGET_COLUMNS:
            plt.plot(merged_df[DATE_COLUMN], merged_df[target], label=target)
        plt.xlabel("Date")
        plt.ylabel("Electric load")
        plt.title("Building Electric Loads Over Time")
        plt.legend()
        plt.tight_layout()
        plt.savefig(RESULTS_FOLDER / "electric_loads_over_time.png", dpi=200)
        plt.close()

    numeric_df = merged_df.select_dtypes(include=np.number)
    correlation_matrix = numeric_df.corr()
    correlation_matrix.to_csv(RESULTS_FOLDER / "correlation_matrix.csv")

    plt.figure(figsize=(16, 12))
    sns.heatmap(correlation_matrix, cmap="coolwarm", center=0)
    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.savefig(RESULTS_FOLDER / "correlation_heatmap.png", dpi=200)
    plt.close()

    feature_df = merged_df.drop(
        columns=TARGET_COLUMNS + [DATE_COLUMN], errors="ignore"
    ).select_dtypes(include=np.number)
    feature_df = feature_df.loc[:, feature_df.nunique(dropna=False) > 1]

    imputer = KNNImputer(n_neighbors=5)
    feature_values = imputer.fit_transform(feature_df)

    vif_data = pd.DataFrame(
        {
            "Feature": feature_df.columns,
            "VIF": [
                variance_inflation_factor(feature_values, index)
                for index in range(feature_values.shape[1])
            ],
        }
    ).sort_values("VIF", ascending=False)

    vif_data.to_csv(RESULTS_FOLDER / "vif_results.csv", index=False)
    print("\nVariance Inflation Factors:")
    print(vif_data.to_string(index=False))


if __name__ == "__main__":
    main()


Dataset shape: (41506, 43)

Data types:
date                             object
mels_S                          float64
lig_S                           float64
mels_N                          float64
hvac_N                          float64
hvac_S                          float64
air_temp_set_1                  float64
air_temp_set_2                  float64
dew_point_temperature_set_1d    float64
relative_humidity_set_1         float64
solar_radiation_set_1           float64
cerc_templogger_1               float64
cerc_templogger_10              float64
cerc_templogger_11              float64
cerc_templogger_12              float64
cerc_templogger_13              float64
cerc_templogger_14              float64
cerc_templogger_15              float64
cerc_templogger_16              float64
cerc_templogger_2               float64
cerc_templogger_3               float64
cerc_templogger_4               float64
cerc_templogger_5               float64
cerc_templogger_6               float64


## 2. Lasso feature selection

In [2]:
"""Lasso feature selection for the five electric-load targets."""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


DATA_PATH = Path("../data/processed/merged_using_date_df.csv")
RESULTS_FOLDER = Path("../results")
DATE_COLUMN = "date"
TARGET_COLUMNS = ["mels_S", "lig_S", "mels_N", "hvac_N", "hvac_S"]


def main() -> None:
    merged_df = pd.read_csv(DATA_PATH)
    RESULTS_FOLDER.mkdir(exist_ok=True)

    if DATE_COLUMN in merged_df.columns:
        merged_df[DATE_COLUMN] = pd.to_datetime(
            merged_df[DATE_COLUMN], errors="coerce"
        )
        merged_df["day_of_week"] = merged_df[DATE_COLUMN].dt.dayofweek
        merged_df["month"] = merged_df[DATE_COLUMN].dt.month
        merged_df["quarter"] = merged_df[DATE_COLUMN].dt.quarter
        merged_df["year"] = merged_df[DATE_COLUMN].dt.year

    X = merged_df.drop(
        columns=TARGET_COLUMNS + [DATE_COLUMN], errors="ignore"
    ).select_dtypes(include=np.number)
    y = merged_df[TARGET_COLUMNS]

    combined = pd.concat([X, y], axis=1).replace([np.inf, -np.inf], np.nan)
    combined = combined.dropna(how="all")
    imputer = KNNImputer(n_neighbors=5)
    imputed = pd.DataFrame(
        imputer.fit_transform(combined),
        columns=combined.columns,
        index=combined.index,
    )
    X = imputed[X.columns]
    y = imputed[TARGET_COLUMNS]

    X_train, _, y_train, _ = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    selected_rows = []
    for target_column in TARGET_COLUMNS:
        lasso_model = Lasso(alpha=0.1)
        lasso_model.fit(X_train_scaled, y_train[target_column])

        selected = X_train.columns[np.abs(lasso_model.coef_) > 0]
        print(f"Selected features for {target_column}:")
        print(list(selected))
        print()

        for feature, coefficient in zip(X_train.columns, lasso_model.coef_):
            if coefficient != 0:
                selected_rows.append(
                    {
                        "Target": target_column,
                        "Feature": feature,
                        "Coefficient": coefficient,
                    }
                )

    pd.DataFrame(selected_rows).to_csv(
        RESULTS_FOLDER / "lasso_selected_features.csv", index=False
    )


if __name__ == "__main__":
    main()


Selected features for mels_S:
['solar_radiation_set_1', 'cerc_templogger_14', 'cerc_templogger_5', 'cerc_templogger_9', 'rtu_004_fltrd_sa_flow_tn', 'day_of_week', 'quarter', 'year']

Selected features for lig_S:
['solar_radiation_set_1', 'cerc_templogger_14', 'cerc_templogger_15', 'cerc_templogger_2', 'cerc_templogger_5', 'rtu_004_fltrd_sa_flow_tn', 'day_of_week', 'year']

Selected features for mels_N:
['solar_radiation_set_1', 'cerc_templogger_10', 'cerc_templogger_14', 'cerc_templogger_2', 'cerc_templogger_5', 'cerc_templogger_9', 'day_of_week', 'quarter', 'year']

Selected features for hvac_N:
['air_temp_set_2', 'solar_radiation_set_1', 'cerc_templogger_1', 'cerc_templogger_10', 'cerc_templogger_12', 'cerc_templogger_14', 'cerc_templogger_15', 'cerc_templogger_2', 'cerc_templogger_5', 'cerc_templogger_6', 'cerc_templogger_8', 'cerc_templogger_9', 'rtu_001_sat_sp_tn', 'rtu_002_sat_sp_tn', 'rtu_001_fltrd_sa_flow_tn', 'rtu_002_fltrd_sa_flow_tn', 'rtu_004_fltrd_sa_flow_tn', 'aru_001_hwr

## 3. Regression-model comparison

In [3]:
"""Train and evaluate the regression models used in the coursework."""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import KNNImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR


DATA_PATH = Path("../data/processed/merged_using_date_df.csv")
RESULTS_FOLDER = Path("../results")
DATE_COLUMN = "date"
TARGET_COLUMNS = ["mels_S", "lig_S", "mels_N", "hvac_N", "hvac_S"]


def metric_row(model_name: str, target: str, actual, predicted) -> dict:
    return {
        "Model": model_name,
        "Target": target,
        "MSE": mean_squared_error(actual, predicted),
        "MAE": mean_absolute_error(actual, predicted),
        "R2": r2_score(actual, predicted),
    }


def main() -> None:
    merged_df = pd.read_csv(DATA_PATH)
    RESULTS_FOLDER.mkdir(exist_ok=True)

    if DATE_COLUMN in merged_df.columns:
        merged_df[DATE_COLUMN] = pd.to_datetime(
            merged_df[DATE_COLUMN], errors="coerce"
        )
        merged_df["day_of_week"] = merged_df[DATE_COLUMN].dt.dayofweek
        merged_df["month"] = merged_df[DATE_COLUMN].dt.month
        merged_df["quarter"] = merged_df[DATE_COLUMN].dt.quarter
        merged_df["year"] = merged_df[DATE_COLUMN].dt.year

    X = merged_df.drop(
        columns=TARGET_COLUMNS + [DATE_COLUMN], errors="ignore"
    ).select_dtypes(include=np.number)
    y = merged_df[TARGET_COLUMNS]

    combined = pd.concat([X, y], axis=1).replace([np.inf, -np.inf], np.nan)
    combined = combined.dropna(how="all")
    imputer = KNNImputer(n_neighbors=5)
    imputed = pd.DataFrame(
        imputer.fit_transform(combined),
        columns=combined.columns,
        index=combined.index,
    )
    X = imputed[X.columns]
    y = imputed[TARGET_COLUMNS]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    results = []

    # Linear Regression without polynomial features
    for target in TARGET_COLUMNS:
        model = LinearRegression()
        model.fit(X_train, y_train[target])
        prediction = model.predict(X_test)
        results.append(
            metric_row("Linear Regression", target, y_test[target], prediction)
        )

    # Linear Regression with degree-2 polynomial features
    polynomial = PolynomialFeatures(degree=2)
    X_train_poly = polynomial.fit_transform(X_train)
    X_test_poly = polynomial.transform(X_test)

    for target in TARGET_COLUMNS:
        model = LinearRegression()
        model.fit(X_train_poly, y_train[target])
        prediction = model.predict(X_test_poly)
        results.append(
            metric_row(
                "Polynomial Linear Regression", target, y_test[target], prediction
            )
        )

    # Random Forest Regression
    for target in TARGET_COLUMNS:
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=50,
            random_state=42,
        )
        model.fit(X_train, y_train[target])
        prediction = model.predict(X_test)
        results.append(
            metric_row("Random Forest", target, y_test[target], prediction)
        )

    # Support Vector Regression with standardised predictor variables
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for target in TARGET_COLUMNS:
        model = SVR(kernel="rbf")
        model.fit(X_train_scaled, y_train[target])
        prediction = model.predict(X_test_scaled)
        results.append(metric_row("SVR", target, y_test[target], prediction))

    # Gradient Boosting Regression
    for target in TARGET_COLUMNS:
        model = GradientBoostingRegressor()
        model.fit(X_train, y_train[target])
        prediction = model.predict(X_test)
        results.append(
            metric_row("Gradient Boosting", target, y_test[target], prediction)
        )

    results_df = pd.DataFrame(results)
    results_df.to_csv(RESULTS_FOLDER / "model_comparison.csv", index=False)

    for model_name in results_df["Model"].unique():
        print(f"\n{model_name}")
        print(results_df[results_df["Model"] == model_name].to_string(index=False))


if __name__ == "__main__":
    main()



Linear Regression
            Model Target       MSE      MAE       R2
Linear Regression mels_S  2.451216 0.991277 0.500114
Linear Regression  lig_S  1.735692 0.959441 0.275254
Linear Regression mels_N  4.842287 1.312969 0.697629
Linear Regression hvac_N 18.994223 2.829402 0.365850
Linear Regression hvac_S 45.547608 4.929653 0.718343

Polynomial Linear Regression
                       Model Target       MSE      MAE       R2
Polynomial Linear Regression mels_S  0.960351 0.576675 0.804152
Polynomial Linear Regression  lig_S  1.198235 0.657711 0.499672
Polynomial Linear Regression mels_N  2.417531 0.892923 0.849040
Polynomial Linear Regression hvac_N  6.951392 1.552598 0.767918
Polynomial Linear Regression hvac_S 19.544998 2.838218 0.879138

Random Forest
        Model Target      MSE      MAE       R2
Random Forest mels_S 0.143825 0.166906 0.970669
Random Forest  lig_S 0.499852 0.360843 0.791285
Random Forest mels_N 0.586349 0.473928 0.963386
Random Forest hvac_N 2.341740 0.756872 0.9

## 4. Manual `mels_S` polynomial experiment

In [4]:
"""Manual feature experiment for the mels_S target."""

from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures


DATA_PATH = Path("../data/processed/merged_using_date_df.csv")

SELECTED_FEATURES = [
    "dew_point_temperature_set_1d",
    "solar_radiation_set_1",
    "cerc_templogger_10",
    "cerc_templogger_14",
    "cerc_templogger_2",
    "cerc_templogger_3",
    "cerc_templogger_4",
    "cerc_templogger_5",
    "cerc_templogger_9",
    "rtu_002_sat_sp_tn",
    "rtu_004_fltrd_sa_flow_tn",
    "aru_001_cws_temp",
]
TARGET = "mels_S"


def print_metrics(name: str, actual, predicted) -> None:
    print(name)
    print("Mean Squared Error:", mean_squared_error(actual, predicted))
    print("Mean Absolute Error:", mean_absolute_error(actual, predicted))
    print("R-squared:", r2_score(actual, predicted))
    print()


def main() -> None:
    merged_df = pd.read_csv(DATA_PATH)

    missing_columns = [
        column for column in SELECTED_FEATURES + [TARGET]
        if column not in merged_df.columns
    ]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    data = merged_df[SELECTED_FEATURES + [TARGET]].dropna()
    X = data[SELECTED_FEATURES]
    y = data[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model_without_poly = LinearRegression()
    model_without_poly.fit(X_train, y_train)
    prediction_without_poly = model_without_poly.predict(X_test)
    print_metrics(
        "Linear Regression without Polynomial Features",
        y_test,
        prediction_without_poly,
    )

    polynomial = PolynomialFeatures(degree=2)
    X_train_poly = polynomial.fit_transform(X_train)
    X_test_poly = polynomial.transform(X_test)

    model_with_poly = LinearRegression()
    model_with_poly.fit(X_train_poly, y_train)
    prediction_with_poly = model_with_poly.predict(X_test_poly)
    print_metrics(
        "Linear Regression with Polynomial Features",
        y_test,
        prediction_with_poly,
    )


if __name__ == "__main__":
    main()


Linear Regression without Polynomial Features
Mean Squared Error: 2.7311563681424373
Mean Absolute Error: 1.007146091576936
R-squared: 0.44302477871505086

Linear Regression with Polynomial Features
Mean Squared Error: 1.7102367743366886
Mean Absolute Error: 0.7844447005524874
R-squared: 0.6512248376010392

